<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Feature Detection and Object Tracking — Implementation</b></h1>
</div>

## Setup — Environment and Configuration


In [1]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(
    precision=4,
    suppress=True,
)
OUTPUT_DIR = Path("../outputs/figures")
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"OpenCV version: {cv2.__version__}")

OpenCV version: 5.0.0


## 1. Validate the Input Video and Output Paths


In [ ]:
DATA_DIR = Path("../data")
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv"}

discovered_video_paths = sorted(
    path for path in DATA_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
)

if not discovered_video_paths:
    raise FileNotFoundError(
        f"No supported video found in: {DATA_DIR}"
    )
VIDEO_INDEX = 0
VIDEO_PATH = discovered_video_paths[VIDEO_INDEX]

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"Videos discovered: {len(discovered_video_paths)}")
print(f"Selected video   : {VIDEO_PATH.name}")
print(f"Figures directory: {OUTPUT_DIR}")

## 2. Define the Initial Bounding Box and Tracking Parameters


In [3]:
ROW = 24

COL = 46

HEIGHT = 170
WIDTH = 160
MAX_FEATURES = 1000
RANSAC_THRESHOLD = 5.0
MIN_MATCHES = 4
MIN_INLIERS = 4

bbox = np.float32(
    [
        [COL, ROW],
        [COL + WIDTH, ROW],
        [COL + WIDTH, ROW + HEIGHT],
        [COL, ROW + HEIGHT],
    ]
).reshape(-1, 1, 2)
if bbox.shape != (4, 1, 2):
    raise ValueError(
        f"Unexpected bounding-box shape: {bbox.shape}"
    )

print("Initial bounding box corners:")
print(bbox.reshape(-1, 2))
print(
    f"ORB features: {MAX_FEATURES} | "
    f"RANSAC threshold: {RANSAC_THRESHOLD:.1f} px"
)

Initial bounding box corners:
[[ 46.  24.]
 [206.  24.]
 [206. 194.]
 [ 46. 194.]]
ORB features: 1000 | RANSAC threshold: 5.0 px


## 3. Initialize ORB and the Hamming-Distance Matcher


In [4]:
orb = cv2.ORB_create(
    nfeatures=MAX_FEATURES,
)
matcher = cv2.BFMatcher(
    cv2.NORM_HAMMING,

crossCheck=True,
)

print("ORB detector initialized.")
print("BFMatcher metric: Hamming | crossCheck=True")

ORB detector initialized.
BFMatcher metric: Hamming | crossCheck=True


## 4. Read and Validate the Reference Frame


In [5]:
cap = cv2.VideoCapture(
    str(VIDEO_PATH)
)
if not cap.isOpened():
    raise RuntimeError(
        f"Could not open video: {VIDEO_PATH}"
    )
reported_frame_count = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)

ret, reference_frame = cap.read()
cap.release()
if not ret or reference_frame is None:
    raise RuntimeError(
        "Could not read the first video frame."
    )

reference_gray_frame = cv2.cvtColor(
    reference_frame,
    cv2.COLOR_BGR2GRAY,
)

frame_height, frame_width = (
    reference_gray_frame.shape
)
if (
    ROW < 0
    or COL < 0
    or ROW + HEIGHT > frame_height
    or COL + WIDTH > frame_width
):
    raise ValueError(
        "The initial bounding box lies outside "
        "the reference frame."
    )

print(
    f"Reference frame shape: "
    f"{reference_frame.shape}"
)
print(
    f"Reported video frames: "
    f"{reported_frame_count}"
)

Reference frame shape: (240, 320, 3)
Reported video frames: 851


## 5. Detect Reference ORB Features Inside the Object Region


In [6]:
reference_region_mask = np.zeros(
    reference_gray_frame.shape,
    dtype=np.uint8,
)

reference_region_mask[
    ROW:ROW + HEIGHT,
    COL:COL + WIDTH,
] = 255

reference_keypoints, reference_descriptors = (
    orb.detectAndCompute(
        reference_gray_frame,
        reference_region_mask,
    )
)
if (
    reference_descriptors is None
    or len(reference_keypoints) < MIN_MATCHES
):
    raise RuntimeError(
        "Not enough ORB features were detected "
        "inside the initial bounding box."
    )
if (
    reference_descriptors.ndim != 2
    or reference_descriptors.shape[1] != 32
):
    raise ValueError(
        "Unexpected ORB descriptor shape: "
        f"{reference_descriptors.shape}"
    )

print(
    f"Reference keypoints detected: "
    f"{len(reference_keypoints)}"
)
print(
    f"Descriptor shape: "
    f"{reference_descriptors.shape}"
)

Reference keypoints detected: 868
Descriptor shape: (868, 32)


## 6. Visualize the Reference Object and ORB Keypoints


In [7]:
reference_with_bbox = (
    reference_frame.copy()
)

cv2.polylines(
    reference_with_bbox,
    [np.int32(bbox)],
    True,
    (0, 255, 0),
    3,
    cv2.LINE_AA,
)

reference_feature_visualization = cv2.drawKeypoints(
    reference_with_bbox,
    reference_keypoints,
    None,
    flags=(
        cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
    ),
)

reference_feature_visualization_rgb = cv2.cvtColor(
    reference_feature_visualization,
    cv2.COLOR_BGR2RGB,
)

fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.imshow(
    reference_feature_visualization_rgb
)
ax.set_title(
    "Reference Object and ORB Keypoints"
)
ax.axis("off")

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "reference_orb_keypoints.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 7. Define Frame Matching and RANSAC Homography Estimation


In [ ]:
def match_features_and_estimate_homography(
    gray_frame,
    reference_keypoints,
    reference_descriptors,
    orb_detector,
    descriptor_matcher,
    min_matches=MIN_MATCHES,
    min_inliers=MIN_INLIERS,
    ransac_threshold=RANSAC_THRESHOLD,
):
    keypoints, descriptors = (
        orb_detector.detectAndCompute(
            gray_frame,
            None,
        )
    )
    if (
        descriptors is None
        or len(keypoints) < min_matches
    ):
        return {
            "success": False,
            "reason": "insufficient_current_features",
            "num_matches": 0,
            "num_inliers": 0,
        }
    matches = descriptor_matcher.match(
        reference_descriptors,
        descriptors,
    )
    matches = sorted(
        matches,
        key=lambda match: match.distance,
    )
    if len(matches) < min_matches:
        return {
            "success": False,
            "reason": "insufficient_matches",
            "num_matches": len(matches),
            "num_inliers": 0,
        }
    matched_reference_points = np.float32(
        [
            reference_keypoints[
                match.queryIdx
            ].pt
            for match in matches
        ]
    ).reshape(-1, 2)
    matched_current_points = np.float32(
        [
            keypoints[
                match.trainIdx
            ].pt
            for match in matches
        ]
    ).reshape(-1, 2)
    homography, inlier_mask = cv2.findHomography(
        matched_reference_points,
        matched_current_points,
        cv2.RANSAC,
        ransac_threshold,
    )
    if (
        homography is None
        or inlier_mask is None
        or homography.shape != (3, 3)
        or not np.all(
            np.isfinite(homography)
        )
    ):
        return {
            "success": False,
            "reason": "homography_failed",
            "num_matches": len(matches),
            "num_inliers": 0,
        }

    inlier_mask = (
        inlier_mask
        .ravel()
        .astype(bool)
    )
    num_inliers = int(
        inlier_mask.sum()
    )
    if num_inliers < min_inliers:
        return {
            "success": False,
            "reason": "insufficient_inliers",
            "num_matches": len(matches),
            "num_inliers": num_inliers,
        }

    return {
        "success": True,
        "reason": None,
        "keypoints": keypoints,
        "matches": matches,
        "homography": homography,
        "inlier_mask": inlier_mask,
        "num_matches": len(matches),
        "num_inliers": num_inliers,
    }
print(
    "Frame matching and homography "
    "estimation function defined."
)

## 8. Track the Object Throughout the Video


In [ ]:
cap = cv2.VideoCapture(
    str(VIDEO_PATH)
)
if not cap.isOpened():
    raise RuntimeError(
        f"Could not open video: {VIDEO_PATH}"
    )
frame_tracking_results = []

current_frame_index = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if current_frame_index == 0:
        current_frame_index += 1
        continue

    gray_frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY,
    )
    result = match_features_and_estimate_homography(
        gray_frame=gray_frame,
        reference_keypoints=reference_keypoints,
        reference_descriptors=reference_descriptors,
        orb_detector=orb,
        descriptor_matcher=matcher,
    )
    tracking_summary = {
        "frame_index": current_frame_index,
        "success": result["success"],
        "reason": result["reason"],
        "num_matches": result["num_matches"],
        "num_inliers": result["num_inliers"],
    }
    if result["success"]:

        tracked_bounding_box = (
            cv2.perspectiveTransform(
                bbox,
                result["homography"],
            )
        )
        if not np.all(
            np.isfinite(tracked_bounding_box)
        ):
            tracking_summary[
                "success"
            ] = False
            tracking_summary[
                "reason"
            ] = "invalid_projected_bbox"
        else:
            tracking_summary[
                "homography"
            ] = result["homography"]
            tracking_summary[
                "current_bbox"
            ] = tracked_bounding_box

    frame_tracking_results.append(
        tracking_summary
    )

    current_frame_index += 1

cap.release()
successful_tracking_results = [
    result
    for result in frame_tracking_results
    if result["success"]
]
failed_tracking_results = [
    result
    for result in frame_tracking_results
    if not result["success"]
]

print(
    f"Frames processed: "
    f"{len(frame_tracking_results)}"
)
print(
    f"Successful tracking frames: "
    f"{len(successful_tracking_results)}"
)
print(
    f"Failed tracking frames: "
    f"{len(failed_tracking_results)}"
)

## 9. Compute Tracking Summary Metrics


In [10]:
if not successful_tracking_results:
    raise RuntimeError(
        "No frame produced a valid homography. "
        "Check the video, ROI, or matching parameters."
    )
tracked_frame_indices = np.asarray(
    [
        result["frame_index"]
        for result in successful_tracking_results
    ],
    dtype=int,
)
feature_match_counts = np.asarray(
    [
        result["num_matches"]
        for result in successful_tracking_results
    ],
    dtype=int,
)
homography_inlier_counts = np.asarray(
    [
        result["num_inliers"]
        for result in successful_tracking_results
    ],
    dtype=int,
)
homography_inlier_ratios = (
    homography_inlier_counts
    / feature_match_counts
)
success_rate = (
    len(successful_tracking_results)
    / len(frame_tracking_results)
)

print(
    f"Tracking success rate: "
    f"{success_rate:.3f}"
)
print(
    f"Mean matches per successful frame: "
    f"{feature_match_counts.mean():.2f}"
)
print(
    f"Mean RANSAC inliers per successful frame: "
    f"{homography_inlier_counts.mean():.2f}"
)
print(
    f"Mean inlier ratio: "
    f"{homography_inlier_ratios.mean():.3f}"
)
print(
    f"Minimum inlier ratio: "
    f"{homography_inlier_ratios.min():.3f}"
)
print(
    f"Maximum inlier ratio: "
    f"{homography_inlier_ratios.max():.3f}"
)

Tracking success rate: 1.000
Mean matches per successful frame: 381.80
Mean RANSAC inliers per successful frame: 290.49
Mean inlier ratio: 0.701
Minimum inlier ratio: 0.040
Maximum inlier ratio: 0.994


## 10. Visualize Representative Tracking Frames


In [ ]:
def read_video_frame_at_index(
    video_path,
    target_index,
):
    capture = cv2.VideoCapture(
        str(video_path)
    )
    if not capture.isOpened():
        raise RuntimeError(
            f"Could not open video: {video_path}"
        )

    capture.set(
        cv2.CAP_PROP_POS_FRAMES,
        int(target_index),
    )

    ret, frame = capture.read()
    capture.release()
    if not ret or frame is None:
        raise RuntimeError(
            f"Could not read frame {target_index}."
        )

    return frame
n_examples = min(
    6,
    len(successful_tracking_results),
)
selected_result_indices = np.linspace(
    0,
    len(successful_tracking_results) - 1,
    n_examples,
    dtype=int,
)
selected_tracking_results = [
    successful_tracking_results[index]
    for index in selected_result_indices
]

n_cols = 3
n_rows = int(
    np.ceil(
        n_examples / n_cols
    )
)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(15, 5 * n_rows),
)

axes = np.asarray(
    axes
).reshape(-1)

for ax, result in zip(
    axes,
    selected_tracking_results,
):
    frame = read_video_frame_at_index(
        VIDEO_PATH,
        result["frame_index"],
    )

    cv2.polylines(
        frame,
        [
            np.int32(
                result["current_bbox"]
            )
        ],
        True,
        (0, 255, 0),
        3,
        cv2.LINE_AA,
    )

    frame_rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB,
    )

    ax.imshow(
        frame_rgb
    )
    ax.set_title(
        f"Frame {result['current_frame_index']} | "
        f"Inliers: "
        f"{result['num_inliers']}/"
        f"{result['num_matches']}"
    )
    ax.axis("off")

for ax in axes[n_examples:]:
    ax.axis("off")

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "representative_tracking_frames.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 11. Visualize RANSAC Inlier Matches


In [12]:
example_tracking_result = (
    successful_tracking_results[
        len(successful_tracking_results) // 2
    ]
)
example_video_frame = read_video_frame_at_index(
    VIDEO_PATH,
    example_tracking_result["frame_index"],
)

example_gray_frame = cv2.cvtColor(
    example_video_frame,
    cv2.COLOR_BGR2GRAY,
)
example_feature_match_result = (
    match_features_and_estimate_homography(
        gray_frame=example_gray_frame,
        reference_keypoints=reference_keypoints,
        reference_descriptors=reference_descriptors,
        orb_detector=orb,
        descriptor_matcher=matcher,
    )
)
if not example_feature_match_result[
    "success"
]:
    raise RuntimeError(
        "Representative frame could not be "
        "re-matched for visualization."
    )
inlier_match_mask = (
    example_feature_match_result[
        "inlier_mask"
    ]
    .astype(np.uint8)
    .tolist()
)

reference_match_display = (
    reference_frame.copy()
)

cv2.polylines(
    reference_match_display,
    [np.int32(bbox)],
    True,
    (0, 255, 0),
    3,
    cv2.LINE_AA,
)

current_match_display = (
    example_video_frame.copy()
)

cv2.polylines(
    current_match_display,
    [
        np.int32(
            example_tracking_result[
                "current_bbox"
            ]
        )
    ],
    True,
    (0, 255, 0),
    3,
    cv2.LINE_AA,
)
match_visualization = (
    cv2.drawMatches(
        reference_match_display,
        reference_keypoints,
        current_match_display,
        example_feature_match_result[
            "keypoints"
        ],
        example_feature_match_result[
            "matches"
        ],
        None,
        matchColor=(0, 255, 0),
        singlePointColor=None,
        matchesMask=inlier_match_mask,
        flags=(
            cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
        ),
    )
)

match_visualization_rgb = (
    cv2.cvtColor(
        match_visualization,
        cv2.COLOR_BGR2RGB,
    )
)

fig, ax = plt.subplots(
    figsize=(16, 7)
)

ax.imshow(
    match_visualization_rgb
)
ax.set_title(
    "RANSAC Inlier Matches — "
    f"Frame {example_tracking_result['current_frame_index']}"
)
ax.axis("off")

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "ransac_inlier_matches.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 12. Analyze Matches, Inliers, and Inlier Ratio Across the Sequence


In [13]:
fig, ax = plt.subplots(
    figsize=(11, 5)
)
ax.plot(
    tracked_frame_indices,
    feature_match_counts,
    marker=".",
    label="Descriptor matches",
)

ax.plot(
    tracked_frame_indices,
    homography_inlier_counts,
    marker=".",
    label="RANSAC inliers",
)

ax.set_title(
    "Feature Matches and RANSAC Inliers by Frame"
)
ax.set_xlabel(
    "Frame index"
)
ax.set_ylabel(
    "Count"
)
ax.grid(
    alpha=0.25
)
ax.legend()

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "matches_and_inliers_by_frame.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

fig, ax = plt.subplots(
    figsize=(11, 5)
)

ax.plot(
    tracked_frame_indices,
    homography_inlier_ratios,
    marker=".",
)

ax.set_title(
    "RANSAC Inlier Ratio by Frame"
)
ax.set_xlabel(
    "Frame index"
)
ax.set_ylabel(
    "Inlier ratio"
)
ax.set_ylim(
    0,
    1.05,
)
ax.grid(
    alpha=0.25
)

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "inlier_ratio_by_frame.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 13. Run Numerical and Output-file Validation Checks


In [14]:
if (
    reference_descriptors is None
    or reference_descriptors.shape[0]
    < MIN_MATCHES
    or reference_descriptors.shape[1]
    != 32
):
    raise ValueError(
        "Invalid reference ORB descriptors."
    )
if len(frame_tracking_results) == 0:
    raise ValueError(
        "No video frames were processed "
        "after the reference frame."
    )
if (
    reported_frame_count > 0
    and len(frame_tracking_results)
    != reported_frame_count - 1
):
    raise ValueError(
        "Processed-frame count does not match "
        "the video frame count."
    )

for result in successful_tracking_results:

    H = result["homography"]
    tracked_bounding_box = result[
        "current_bbox"
    ]
    if (
        H.shape != (3, 3)
        or not np.all(
            np.isfinite(H)
        )
    ):
        raise ValueError(
            "Invalid homography in "
            f"frame {result['current_frame_index']}."
        )
    if (
        tracked_bounding_box.shape
        != (4, 1, 2)
        or not np.all(
            np.isfinite(tracked_bounding_box)
        )
    ):
        raise ValueError(
            "Invalid transformed bounding box "
            f"in frame {result['current_frame_index']}."
        )
    if (
        result["num_matches"]
        < MIN_MATCHES
        or result["num_inliers"]
        < MIN_INLIERS
        or result["num_inliers"]
        > result["num_matches"]
    ):
        raise ValueError(
            "Invalid correspondence statistics "
            f"in frame {result['current_frame_index']}."
        )
if not np.all(
    np.isfinite(
        homography_inlier_ratios
    )
):
    raise ValueError(
        "Non-finite inlier ratio detected."
    )
if (
    np.any(homography_inlier_ratios < 0.0)
    or np.any(homography_inlier_ratios > 1.0)
):
    raise ValueError(
        "Inlier ratios outside [0, 1]."
    )
REQUIRED_OUTPUTS = [
    "reference_orb_keypoints.png",
    "representative_tracking_frames.png",
    "ransac_inlier_matches.png",
    "matches_and_inliers_by_frame.png",
    "inlier_ratio_by_frame.png",
]
missing_outputs = [
    name
    for name in REQUIRED_OUTPUTS
    if not (
        OUTPUT_DIR / name
    ).exists()
]
if missing_outputs:
    raise FileNotFoundError(
        "Missing outputs: "
        + ", ".join(
            missing_outputs
        )
    )

print(
    "All Feature Detection validation checks passed."
)

All Feature Detection validation checks passed.
